# 第 6 周：分类综合实践与累计测试

**对应主线**：为李宏毅课程 HW2 Classification 建立原创前置能力，不复制官方题目或数据。  
**任务**：在三个合成“设备状态”类别上完成训练、验证、测试、混淆矩阵和错误分析。  
**先修**：前 5 周；需要 `numpy`、`torch`。

最终成果：

1. 独立写出多分类训练循环；
2. 固定 train/validation/test 三份数据；
3. 测试集只在模型确定后使用一次；
4. 找出错误样本并提出可验证改进；
5. 完成前 6 周累计出口测验。


In [ ]:
import copy
import numpy as np
import torch
from torch import nn

np.random.seed(31)
torch.manual_seed(31)


## 1. 创建三类原创合成数据


In [ ]:
rng = np.random.default_rng(31)
centers = np.array([[-2.0, -1.0], [2.0, -0.5], [0.0, 2.2]], dtype="float32")
clouds = []
labels = []

for class_id, center in enumerate(centers):
    clouds.append(rng.normal(loc=center, scale=0.65, size=(180, 2)).astype("float32"))
    labels.append(np.full(180, class_id, dtype="int64"))

x_np = np.vstack(clouds)
y_np = np.concatenate(labels)
order = rng.permutation(len(x_np))
train_idx, val_idx, test_idx = order[:360], order[360:450], order[450:]

x_train, y_train = torch.tensor(x_np[train_idx]), torch.tensor(y_np[train_idx])
x_val, y_val = torch.tensor(x_np[val_idx]), torch.tensor(y_np[val_idx])
x_test, y_test = torch.tensor(x_np[test_idx]), torch.tensor(y_np[test_idx])

print("train/val/test:", x_train.shape, x_val.shape, x_test.shape)


In [ ]:
assert set(train_idx).isdisjoint(set(val_idx))
assert set(train_idx).isdisjoint(set(test_idx))
assert set(val_idx).isdisjoint(set(test_idx))
assert x_train.shape[1] == 2
print("数据划分测试通过 ✅")


## 2. 训练并用验证集选模型

这里保存验证准确率最佳的参数。测试集没有参与参数更新和模型选择。


In [ ]:
def accuracy_from_logits(logits, labels):
    return (logits.argmax(dim=1) == labels).float().mean().item()


torch.manual_seed(31)
model = nn.Sequential(
    nn.Linear(2, 12),
    nn.ReLU(),
    nn.Linear(12, 3),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.025, weight_decay=1e-4)

best_state = None
best_val_accuracy = -1.0
patience = 40
stale_checks = 0

for epoch in range(350):
    model.train()
    optimizer.zero_grad()
    train_logits = model(x_train)
    train_loss = loss_fn(train_logits, y_train)
    train_loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_accuracy = accuracy_from_logits(model(x_val), y_val)
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = copy.deepcopy(model.state_dict())
            stale_checks = 0
        else:
            stale_checks += 1
        if stale_checks >= patience:
            break

model.load_state_dict(best_state)
print("best validation accuracy:", best_val_accuracy)


## 3. 一次性测试与混淆矩阵


In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(x_test)
    test_predictions = test_logits.argmax(dim=1)
    test_accuracy = (test_predictions == y_test).float().mean().item()


def confusion_matrix(predictions, targets, class_count):
    matrix = torch.zeros((class_count, class_count), dtype=torch.int64)
    for target, prediction in zip(targets, predictions):
        matrix[target, prediction] += 1
    return matrix


matrix = confusion_matrix(test_predictions, y_test, class_count=3)
print(f"test accuracy={test_accuracy:.3f}")
print("rows=true, columns=predicted\n", matrix)


In [ ]:
assert test_logits.shape == (90, 3)
assert matrix.shape == (3, 3)
assert matrix.sum().item() == len(y_test)
assert test_accuracy > 0.93
print("综合分类测试通过 ✅")


## 4. 错误分析

不要只看总准确率。列出最不确定的错误样本，检查它们是否靠近类别边界。


In [ ]:
with torch.no_grad():
    probabilities = torch.softmax(test_logits, dim=1)
    confidence = probabilities.max(dim=1).values

wrong = torch.where(test_predictions != y_test)[0]
wrong_sorted = wrong[torch.argsort(confidence[wrong])] if len(wrong) else wrong

for index in wrong_sorted[:5]:
    print({
        "features": x_test[index].tolist(),
        "true": y_test[index].item(),
        "predicted": test_predictions[index].item(),
        "confidence": round(confidence[index].item(), 3),
    })


用三句话写复盘：

1. 错误主要集中在哪两类之间？
2. 是数据重叠、模型容量、训练不稳定还是评估方式造成？
3. 下一次只改一个变量，你会改什么？如何判断它有效？


## 5. 前 6 周累计自动测验

这些断言覆盖容器/函数、shape、矩阵乘法、训练输出和评估。先自己回答，再运行。


In [ ]:
def even_mean(numbers):
    evens = [number for number in numbers if number % 2 == 0]
    return sum(evens) / len(evens) if evens else None


assert even_mean([1, 2, 4, 9]) == 3.0
assert even_mean([1, 3]) is None
assert (torch.ones(8, 4) @ torch.ones(4, 3)).shape == (8, 3)
assert nn.Linear(5, 2)(torch.zeros(7, 5)).shape == (7, 2)
assert abs(torch.softmax(torch.tensor([[1.0, 2.0, 3.0]]), dim=1).sum().item() - 1) < 1e-6
print("累计自动测验：5/5 通过 ✅")


## 常见错误

- 用测试集挑模型后仍称它为“最终测试”；
- 混淆矩阵把行列含义写反；
- 只展示最好数字，不记录种子、划分与失败实验；
- 代码能运行却解释不出输入、输出、loss 和验证结果；
- 没有运行证据就认为作业完成。

## 扩展

- 提高某一类噪声，观察混淆矩阵变化；
- 比较线性模型与当前小网络；
- 将每次实验的配置和结果保存为字典，再输出 Markdown 表格。

**阶段通过条件**：测试准确率超过 93%，全部断言通过；闭卷重写训练循环；完成错误分析和一项单变量改进。正式进入课程作业前，再完成一次 85% 以上的累计闭卷测验。
